In [1]:
import scanpy as sc
import pandas as pd
import anndata as ad
import yaml
import socket
import os
import warnings

In [2]:
Branch_adata = sc.read_h5ad("saved/scanpy/all_donor_annotated_05_wPaga.h5ad")
Branch_adata.X = Branch_adata.layers["counts"]

In [3]:
BranchBCs = Branch_adata.obs_names[Branch_adata.obs["endpoint_Glutamatergic neuron"] == 1].tolist()
Branchtag = "GlutamatergicN"
print(len(BranchBCs))

17972


In [4]:
Branch_adata_glu = Branch_adata[pd.Series(list(set(Branch_adata.obs_names).intersection(set(BranchBCs))))]
Branch_adata_glu

View of AnnData object with n_obs × n_vars = 17972 × 20733
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'donorId', 'percent.mt', 'percent.ribo', 'S.Score', 'G2M.Score', 'Phase', 'old.ident', 'batch', 'condition', 'donor', 'RNA_snn_res.0.5', 'seurat_clusters', 'sctype_pred', 'leiden_0.25', 'endpoint_Glutamatergic neuron', 'endpoint_GABAergic neuron'
    var: 'vf_vst_counts_mean', 'vf_vst_counts_variance', 'vf_vst_counts_variance.expected', 'vf_vst_counts_variance.standardized', 'vf_vst_counts_variable', 'vf_vst_counts_rank', 'var.features', 'var.features.rank', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_seurat'
    uns: 'draw_graph', 'hvg', 'leiden_0.25', 'leiden_0.25_colors', 'leiden_0.25_sizes', 'log1p', 'neighbors', 'paga', 'sctype_pred_colors', 'sctype_pred_sizes'
    obsm: 'X_draw_graph_fa', 'X_harmony', 'X_pca', 'X_umap'
    varm: 'HARMONY', 'PCs', 'harmony_loadings', 'pca_loadings'
    layers: 'counts'
    obsp: 'connectivities', 'dist

In [5]:
Branch_adata_glu.obs["orig.ident"].value_counts()


orig.ident
CL1_viCO    4718
CL1_vCO     3225
CL3_viCO    2828
CL1_cCO     2380
CL3_cCO     1649
CL3_vCO     1578
CL2_cCO      621
CL2_vCO      522
CL2_viCO     451
Name: count, dtype: int64

In [6]:
desired_order = [
    "CL1_cCO",
    "CL2_cCO",
    "CL3_cCO",
    "CL1_vCO",
    "CL2_vCO",
    "CL3_vCO",
    "CL1_viCO",
    "CL2_viCO",
    "CL3_viCO"
]

print(Branch_adata.obs["orig.ident"].value_counts().loc[desired_order])

ct1 = pd.crosstab(
    Branch_adata.obs["orig.ident"],
    Branch_adata.obs["sctype_pred"]
)

ct1 = ct1.loc[desired_order]
ct1


orig.ident
CL1_cCO     3669
CL2_cCO     2574
CL3_cCO     2504
CL1_vCO     3981
CL2_vCO     4223
CL3_vCO     3050
CL1_viCO    5025
CL2_viCO    4354
CL3_viCO    4520
Name: count, dtype: int64


sctype_pred,ChP epithelial,Endothelial,Fibroblast,GABAergic neuron,Glutamatergic neuron,Immature neuron,Microglia,Neuroblast,Radial glia,Roof plate progenitor,Schwann precursor,Smooth muscle
orig.ident,,,,,,,,,,,,
CL1_cCO,0,0,31,833,971,165,0,168,1496,2,2,1
CL2_cCO,215,0,40,123,313,62,0,83,303,1406,25,4
CL3_cCO,1,0,83,522,428,124,3,391,724,197,27,4
CL1_vCO,7,0,55,400,304,292,0,293,2381,243,1,5
CL2_vCO,353,18,890,54,192,66,16,69,362,1808,30,365
CL3_vCO,74,8,325,282,781,272,1,175,382,462,171,117
CL1_viCO,2,0,38,213,637,651,0,474,2970,36,0,4
CL2_viCO,500,16,1121,24,309,32,417,25,130,1359,47,374
CL3_viCO,129,0,418,385,1043,400,3,361,1058,520,121,82


In [7]:
print(Branch_adata_glu.obs["orig.ident"].value_counts().loc[desired_order])

ct2 = pd.crosstab(
    Branch_adata_glu.obs["orig.ident"],
    Branch_adata_glu.obs["sctype_pred"]
)

ct2 = ct2.loc[desired_order]
ct2


orig.ident
CL1_cCO     2380
CL2_cCO      621
CL3_cCO     1649
CL1_vCO     3225
CL2_vCO      522
CL3_vCO     1578
CL1_viCO    4718
CL2_viCO     451
CL3_viCO    2828
Name: count, dtype: int64


sctype_pred,Glutamatergic neuron,Immature neuron,Neuroblast,Radial glia
orig.ident,,,,
CL1_cCO,669,159,60,1492
CL2_cCO,313,62,80,166
CL3_cCO,424,124,388,713
CL1_vCO,302,292,271,2360
CL2_vCO,192,66,69,195
CL3_vCO,776,272,169,361
CL1_viCO,630,651,470,2967
CL2_viCO,309,32,25,85
CL3_viCO,1041,400,352,1035


In [8]:
## HVG on PLCG1

In [9]:
adata=sc.read_h5ad("saved/scanpy/all_donor_annotated_05_wPaga.h5ad")
adata.X = adata.layers["counts"]
print(adata.shape)
mask_donor1 = adata.obs["donor"]=="CL3"
adata_plcg1 = adata[mask_donor1, :].copy()
print(adata_plcg1.shape)

(33900, 20733)
(10074, 20733)


In [10]:
adata_plcg1.var_names_make_unique()

#Subset by cells present only in the astrocyte trajectory
adata_plcg1 = adata_plcg1[pd.Series(list(set(adata_plcg1.obs_names).intersection(set(BranchBCs))))]
print(adata_plcg1.shape)
print(adata_plcg1.obs["condition"].value_counts())
sc.pp.filter_genes(adata_plcg1, min_cells=3)
print(adata_plcg1.X[:10, :10])
sc.pp.normalize_total(adata_plcg1, target_sum=2e4)
print(adata_plcg1.X[:10, :10])
sc.pp.log1p(adata_plcg1)
print(adata_plcg1.X[:10, :10])
sc.pp.highly_variable_genes(adata_plcg1, n_top_genes=2000, batch_key = "condition")
print(adata_plcg1.var["highly_variable_nbatches"].value_counts())

HVG_d1 = adata_plcg1.var[adata_plcg1.var.highly_variable_nbatches >= 1].index.tolist()
print(len(HVG_d1))

del adata

(6055, 20733)
condition
viCO    2828
cCO     1649
vCO     1578
Name: count, dtype: int64


/PUHTI_TYKKY_gU5QJ1k/miniforge/envs/env1/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:284: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (10, 10)>
  Coords	Values
  (2, 7)	1.0
  (3, 4)	1.0
  (4, 4)	1.0
  (4, 8)	1.0
  (5, 4)	1.0
  (9, 7)	1.0
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (10, 10)>
  Coords	Values
  (2, 7)	2.649357530798781
  (3, 4)	2.362111727884729
  (4, 4)	1.732051615138131
  (4, 8)	1.732051615138131
  (5, 4)	1.6899028305872412
  (9, 7)	1.9731649565903708
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (10, 10)>
  Coords	Values
  (2, 7)	1.294551133142007
  (3, 4)	1.2125692669995911
  (4, 4)	1.0050528343331995
  (4, 8)	1.0050528343331995
  (5, 4)	0.9895050705030414
  (9, 7)	1.089627027370902
highly_variable_nbatches
0    17609
1     1258
3     1204
2      565
Name: count, dtype: int64
3027


In [11]:
## HVG on Alz37cl2


In [12]:
adata=sc.read_h5ad("saved/scanpy/all_donor_annotated_05_wPaga.h5ad")
adata.X = adata.layers["counts"]
print(adata.shape)
mask_donor2 = adata.obs["donor"]=="CL1"
adata_alz37 = adata[mask_donor2, :].copy()
print(adata_alz37.shape)

(33900, 20733)
(12675, 20733)


In [13]:
adata_alz37.var_names_make_unique()

#Subset by cells present only in the astrocyte trajectory
adata_alz37 = adata_alz37[pd.Series(list(set(adata_alz37.obs_names).intersection(set(BranchBCs))))]
print(adata_alz37.shape)
print(adata_alz37.obs["condition"].value_counts())

sc.pp.filter_genes(adata_alz37, min_cells=3)
sc.pp.normalize_total(adata_alz37, target_sum=2e4)
sc.pp.log1p(adata_alz37)
sc.pp.highly_variable_genes(adata_alz37, n_top_genes=2000, batch_key = "condition")
print(adata_alz37.var["highly_variable_nbatches"].value_counts())

HVG_d2 = adata_alz37.var[adata_alz37.var.highly_variable_nbatches >= 1].index.tolist()
print(len(HVG_d2))

del adata

(10323, 20733)
condition
viCO    4718
vCO     3225
cCO     2380
Name: count, dtype: int64


/PUHTI_TYKKY_gU5QJ1k/miniforge/envs/env1/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:284: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


highly_variable_nbatches
0    17505
1     1254
3     1154
2      642
Name: count, dtype: int64
3050


In [14]:
## HVG on Alz43cl2

In [15]:
adata=sc.read_h5ad("saved/scanpy/all_donor_annotated_05_wPaga.h5ad")
adata.X = adata.layers["counts"]
print(adata.shape)
mask_donor3 = adata.obs["donor"]=="CL2"
adata_alz43 = adata[mask_donor3, :].copy()
print(adata_alz43.shape)

(33900, 20733)
(11151, 20733)


In [16]:
adata_alz43.var_names_make_unique()

#Subset by cells present only in the astrocyte trajectory
adata_alz43 = adata_alz43[pd.Series(list(set(adata_alz43.obs_names).intersection(set(BranchBCs))))]
print(adata_alz43.shape)
print(adata_alz43.obs["condition"].value_counts())

sc.pp.filter_genes(adata_alz43, min_cells=3)
sc.pp.normalize_total(adata_alz43, target_sum=2e4)
sc.pp.log1p(adata_alz43)
sc.pp.highly_variable_genes(adata_alz43, n_top_genes=2000, batch_key = "condition")
print(adata_alz43.var["highly_variable_nbatches"].value_counts())

HVG_d3 = adata_alz43.var[adata_alz43.var.highly_variable_nbatches >= 1].index.tolist()
print(len(HVG_d3))

del adata

(1594, 20733)
condition
cCO     621
vCO     522
viCO    451
Name: count, dtype: int64


/PUHTI_TYKKY_gU5QJ1k/miniforge/envs/env1/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:284: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


highly_variable_nbatches
0    16979
1     1731
3      987
2      654
Name: count, dtype: int64
3372


In [17]:
print(len(HVG_d1))
print(len(HVG_d2))
print(len(HVG_d3))

print(len(set(HVG_d1) & set(HVG_d2)))
print(len(set(HVG_d1) & set(HVG_d3)))
print(len(set(HVG_d2) & set(HVG_d3)))

HVGoverlap = list(set(HVG_d1+HVG_d2+HVG_d3))
len(HVGoverlap)

3027
3050
3372
1971
1891
1802


5272

In [18]:
External = ["SOX2","PAX6","HES1","EOMES","DCX","NEUROD1","STMN2","MAP2","SLC17A7","TBR1","CUX1"]
HVGoverlap_curated = list(set(HVGoverlap+External))
pd.DataFrame(HVGoverlap_curated, columns = ["HVG"]).to_csv("saved/HVG_list_intersection_Curated_"+Branchtag+"_PagaTest.txt", sep="\t")

In [19]:
print(len(HVGoverlap))

5272
